In [1]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0

from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

✅ Imports successful
✅ CUDA available: False


# BirdCLEF 2026 Inference v6 - 2-Model Ensemble (Perch + EfficientNet-B0)
## Strategy:
- Load 5 Perch folds
- Load 5 EfficientNet-B0 folds
- 2-window ensemble (start + middle)
- Average 20 predictions (10 models × 2 windows)
- Apply per-species thresholds

In [2]:
# === SETUP ===
TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
)

print(f"✅ Config set up")
print(f"✅ Device: {device}")

# === CHECK DATA AVAILABILITY ===
import os
if not os.path.exists(TEST_AUDIO_DIR):
    print(f"\n⚠️  WARNING: Test data directory not found!")
    print(f"   Path: {TEST_AUDIO_DIR}")
    print(f"   This notebook is designed to run on Kaggle where test data is available.")
    print(f"   Running locally will produce incomplete/empty predictions.")
    print(f"   Continuing anyway - submission will contain no predictions.\n")
else:
    print(f"✅ Test audio directory found: {TEST_AUDIO_DIR}")

✅ Config set up
✅ Device: cpu
✅ Test audio directory found: /kaggle/input/competitions/birdclef-2026/test_soundscapes


In [3]:
# === LOAD SPECIES & THRESHOLDS ===
# Try to load from Kaggle path first, fall back to local
species_paths = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json",
    "species.json"
]

species = None
for path in species_paths:
    try:
        with open(path, "r") as f:
            species = json.load(f)
        print(f"✅ Loaded species from: {path}")
        break
    except FileNotFoundError:
        continue

if species is None:
    print("❌ ERROR: Could not load species.json from any path!")
    raise FileNotFoundError("species.json not found")

# Try to load thresholds from Kaggle path, fall back to defaults
thresholds_path = "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/optimal_thresholds_v6.json"
try:
    with open(thresholds_path, "r") as f:
        optimal_thresholds = json.load(f)
    print(f"✅ Loaded per-species thresholds")
except FileNotFoundError:
    print(f"⚠️  Using default threshold (0.5) for all species")
    optimal_thresholds = {sp: 0.5 for sp in species}

idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

print(f"✅ Loaded {n_classes} species")
print(f"First 10 species: {species[:10]}")

✅ Loaded species from: /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json
✅ Loaded per-species thresholds
✅ Loaded 206 species
First 10 species: ['1161364', '116570', '1176823', '1595929', '209233', '22930', '22956', '22961', '22967', '22973']


In [4]:
# === HELPER FUNCTIONS ===
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")

✅ Helper functions defined


In [5]:
# === PERCH ARCHITECTURE ===
class PerchAudio(nn.Module):
    """Lightweight CNN optimized for bird audio"""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.head = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

print("✅ Perch model defined")

✅ Perch model defined


In [6]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("✅ EfficientNet-B0 model defined")

✅ EfficientNet-B0 model defined


In [7]:
# === LOAD ALL TRAINED MODELS ===
print("Loading all 10 trained models (5 folds × 2 architectures)...")

perch_models = []
efficientnet_models = []

for fold_idx in range(5):
    # Load Perch
    perch_model = PerchAudio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    perch_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/perch_v6_fold_{fold_idx}.pt", map_location=device))
    perch_model.eval()
    perch_models.append(perch_model)
    
    # Load EfficientNet-B0
    eff_model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    eff_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/efficientnet_v6_fold_{fold_idx}.pt", map_location=device))
    eff_model.eval()
    efficientnet_models.append(eff_model)

print(f"✅ Loaded {len(perch_models)} Perch folds")
print(f"✅ Loaded {len(efficientnet_models)} EfficientNet-B0 folds")

Loading all 10 trained models (5 folds × 2 architectures)...
✅ Loaded 5 Perch folds
✅ Loaded 5 EfficientNet-B0 folds


In [8]:
# === TEST AUDIO DATASET (2-WINDOW STRATEGY) ===
class TestAudioDataset(Dataset):
    def __init__(self, audio_path, cfg):
        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
        except:
            print(f"Failed to load: {audio_path}")
            y, sr0 = np.zeros(cfg["sr"] * cfg["seconds"]), cfg["sr"]
        
        if sr0 != cfg["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=cfg["sr"])
        
        y = fixed_length_mono(y, cfg["sr"], cfg["seconds"])
        self.mel = logmel_from_wave(y, cfg["sr"])
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1
    
    def __len__(self):
        # Two windows per audio
        return 2
    
    def __getitem__(self, idx):
        T = self.mel.shape[1]
        W = self.win_frames
        
        # Two-window: start and middle
        if idx == 0:
            start = 0
        else:
            start = max(0, (T - W) // 2)
        
        if T <= W:
            pad = np.zeros((self.mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([self.mel, pad], axis=1)
        else:
            mel = self.mel[:, start:start + W]
        
        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)
        x = torch.from_numpy(mel)[None, ...]
        return x.float()

print("✅ Test dataset class defined")

✅ Test dataset class defined


In [9]:
# === PREDICTION FUNCTION ===
def get_predictions_for_audio(audio_path):
    """Get ensemble predictions from 10 models × 2 windows"""
    
    dataset = TestAudioDataset(audio_path, CFG)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    
    all_preds = np.zeros((n_classes,), dtype=np.float32)
    
    with torch.no_grad():
        for window_idx, x in enumerate(loader):
            x = x.to(device)
            
            # Get predictions from all Perch folds
            for perch_model in perch_models:
                logits = perch_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                all_preds += probs
            
            # Get predictions from all EfficientNet folds
            for eff_model in efficientnet_models:
                logits = eff_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                all_preds += probs
    
    # Average over all windows and models (2 windows × 5 folds × 2 architectures = 20 predictions)
    ensemble_preds = all_preds / (2 * 5 * 2)
    
    return ensemble_preds

print("✅ Prediction function defined")

✅ Prediction function defined


In [10]:
# === GENERATE PREDICTIONS ===
test_dir = Path(TEST_AUDIO_DIR)
test_files = sorted(test_dir.glob("*.ogg")) if test_dir.exists() else []

print(f"Found {len(test_files)} test files in {TEST_AUDIO_DIR}")

if len(test_files) == 0:
    print(f"\n⚠️  WARNING: No test audio files found!")
    print(f"   This notebook requires the Kaggle test_soundscapes directory.")
    print(f"   Running locally will produce an EMPTY submission.")
    print(f"   Submission CSV will have proper structure but NO PREDICTIONS.\n")

predictions = {}

for audio_path in tqdm(test_files, desc="Generating predictions"):
    filename = audio_path.stem
    ensemble_preds = get_predictions_for_audio(str(audio_path))
    predictions[filename] = ensemble_preds

print(f"\n✅ Generated predictions for {len(predictions)} test files")

Found 0 test files in /kaggle/input/competitions/birdclef-2026/test_soundscapes

⚠️  WARNING: No test audio files found!
   This notebook requires the Kaggle test_soundscapes directory.
   Running locally will produce an EMPTY submission.
   Submission CSV will have proper structure but NO PREDICTIONS.



Generating predictions: 0it [00:00, ?it/s]


✅ Generated predictions for 0 test files


In [ ]:
# === APPLY PER-SPECIES THRESHOLDS & CREATE SUBMISSION ===
submission_data = []

if len(predictions) == 0:
    print("⚠️  No predictions available. Creating empty submission DataFrame.")
else:
    for filename, preds in predictions.items():
        row = {"row_id": filename}
        
        for j, sp in enumerate(species):
            threshold = optimal_thresholds.get(sp, 0.5)
            prediction = 1 if preds[j] >= threshold else 0
            row[sp] = prediction
        
        submission_data.append(row)

if len(submission_data) == 0:
    # Create empty DataFrame with correct structure
    cols = ["row_id"] + species
    submission_df = pd.DataFrame(columns=cols)
    print("Created empty submission DataFrame with correct structure")
else:
    submission_df = pd.DataFrame(submission_data)

print(f"\n✅ Submission DataFrame created")
print(f"Shape is: {submission_df.shape}")
if len(submission_df) > 0:
    print(f"\nFirst 5 rows (first 10 columns):")
    print(submission_df.iloc[:5, :10])
else:
    print(f"DataFrame is empty - no test data was available")
    print(f"Columns: {list(submission_df.columns[:10])}... (and {len(submission_df.columns) - 10} more)")

⚠️  No predictions available. Creating empty submission DataFrame.
Created empty submission DataFrame with correct structure

✅ Submission DataFrame created
Shape: (0, 207)
DataFrame is empty - no test data was available
Columns: ['row_id', '1161364', '116570', '1176823', '1595929', '209233', '22930', '22956', '22961', '22967']... (and 197 more)


In [12]:
# === SAVE SUBMISSION ===
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"✅ Submission saved to: /kaggle/working/submission.csv")
print(f"\n📊 SUBMISSION SUMMARY:")
print(f"  Total predictions: {len(submission_df)}")
# Extract actual species columns (all except 'row_id')
species_cols = [col for col in submission_df.columns if col != 'row_id']
print(f"  Total species: {len(species_cols)}")
if len(submission_df) > 0:
    print(f"  Avg positive predictions per test: {submission_df[species_cols].sum(axis=1).mean():.2f}")
print(f"\n✅ INFERENCE COMPLETE!")
print(f"\n📈 ENSEMBLE STRATEGY:")
print(f"  - 5 Perch folds")
print(f"  - 5 EfficientNet-B0 folds")
print(f"  - 2 windows per test audio (start and middle)")
print(f"  - Total: 20 predictions averaged per test file (vs 30 in v5)")
print(f"  - Per-species optimal thresholds applied")

✅ Submission saved to: /kaggle/working/submission.csv

📊 SUBMISSION SUMMARY:
  Total predictions: 0
  Total species: 206

✅ INFERENCE COMPLETE!

📈 ENSEMBLE STRATEGY:
  - 5 Perch folds
  - 5 EfficientNet-B0 folds
  - 2 windows per test audio (start and middle)
  - Total: 20 predictions averaged per test file (vs 30 in v5)
  - Per-species optimal thresholds applied
